# 07 — Full RAG Pipeline End-to-End

**Pipeline position:** Integrates all previous modules (01-06).

**Purpose:** Demonstrate the complete RAG pipeline from query to cited answer,
including hybrid retrieval, reranking, LLM generation, and post-processing.

**Learning objectives:**
- Initialize and run `RAGPipeline` with a single method call
- Inspect the structured output: answer, citations, images, latency
- Use multi-turn conversation support
- Analyze per-module latency breakdown

## Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
from src.pipeline import RAGPipeline

## Initialize RAGPipeline

This loads BM25, Chroma, BGE-M3 encoder, and the BGE reranker.
First run may take 30-60s to load all models.

In [ ]:
pipeline = RAGPipeline()

## Single query — full result

The `answer()` method returns a dict with answer, citations, images, latency, and retrieved docs.

In [ ]:
result = pipeline.answer('What signals are required for T cell activation?')

print('=== Answer ===')
print(result['answer'])
print(f'\n=== Citations ===')
print(f'Pages:    {result["cite_pages"]}')
print(f'Sources:  {result["cite_sources"]}')
print(f'Chapters: {result["cite_chapters"]}')
print(f'Images:   {len(result["related_images"])} figure(s)')
print(f'\n=== Latency ===')
for module, ms in sorted(result['latency_ms'].items(), key=lambda x: -x[1]):
    print(f'  {module:12s}: {ms:7.1f} ms')
print(f'  {"TOTAL":12s}: {sum(result["latency_ms"].values()):7.1f} ms')

## Inspect retrieved documents

The pipeline returns the top-K reranked documents that were used as LLM context.

In [ ]:
for i, doc in enumerate(result['retrieved_docs'], 1):
    meta = doc.metadata
    print(f'[{i}] Source: {meta.get("source_file", "?")}')
    print(f'    Chapter: {meta.get("chapter", "?")} | Page: {meta.get("page", "?")}')
    print(f'    Text: {doc.page_content[:120]}...\n')

## Multi-turn conversation

The pipeline maintains chat history. Follow-up questions use previous context.

In [ ]:
pipeline.reset_history()

r1 = pipeline.answer('What are T helper cells?')
print('Q1: What are T helper cells?')
print(f'A1: {r1["answer"][:200]}...\n')

r2 = pipeline.answer('How do they interact with B cells?')
print('Q2: How do they interact with B cells?')
print(f'A2: {r2["answer"][:200]}...\n')

print(f'Chat history length: {len(pipeline.get_history())} messages')

## Latency breakdown visualization

Stacked bar showing where time is spent in the pipeline.

In [ ]:
latency = result['latency_ms']
modules = sorted(latency.keys(), key=lambda m: latency[m], reverse=True)
values = [latency[m] for m in modules]
colors = plt.cm.tab10(np.linspace(0, 1, len(modules)))

fig, ax = plt.subplots(figsize=(10, 4))
left = 0
for m, v, c in zip(modules, values, colors):
    ax.barh('Pipeline', v, left=left, color=c, label=f'{m} ({v:.0f}ms)', height=0.5)
    left += v
ax.set_xlabel('Latency (ms)')
ax.set_title(f'Per-Module Latency (Total: {sum(values):.0f} ms)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Summary

**Key takeaways:**
- `RAGPipeline.answer()` orchestrates the entire flow in one call
- Structured output provides answer, citations, images, and latency
- Multi-turn history enables contextual follow-up questions
- LLM generation typically dominates latency

**Next:** `08_build_train_data.ipynb` — Generate training data for fine-tuning.